# SpaNCy-Shift demo — train, normalize, inspect histograms, save `.h5ad`

Runs `spancy_shift_spatial.py` on the PRAD-CyCIF dataset (1.76 M cells, 20 markers, 20 samples,
7 batches): Stage 1 (analytic shift) → Stage 2 (region graph neural network) → output at α = 0.6.

**Just run all cells** — `spancy_shift_spatial.py` is downloaded from GitHub next to this notebook
(Colab: `/content`, open it from the file browser) and the data from Zenodo.
A GPU runtime is recommended (Colab: *Runtime → Change runtime type → GPU*): preparation takes about
one minute and training about 15 minutes. GPU training is not bit-reproducible, so a re-run gives
slightly different numbers.

## 1. Setup

In [ ]:
# Colab: installs torch-geometric for Colab's torch + CUDA version and the tested pins.
# Locally: skip this cell and run `pip install -r requirements.txt` instead.
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q "pandas<3" "anndata<0.11"
print('Done.')

In [ ]:
# Downloads spancy_shift_spatial.py from GitHub next to this notebook (Colab: /content), so you can
# open it in the file browser, and imports it from there. Any older copy is overwritten.
# SPANCY_REF = branch, tag or commit to use; 'master' is the released version.
SPANCY_REF = 'master'

import os
import importlib
import urllib.request

MODULE_URL = f'https://raw.githubusercontent.com/teepeen/SpaNCy-Shift/{SPANCY_REF}/spancy_shift_spatial.py'
urllib.request.urlretrieve(MODULE_URL, 'spancy_shift_spatial.py')
import spancy_shift_spatial
importlib.reload(spancy_shift_spatial)
assert os.path.dirname(os.path.abspath(spancy_shift_spatial.__file__)) == os.getcwd(), \
    f'imported a different copy: {spancy_shift_spatial.__file__}'
print(f'spancy_shift_spatial.py ({SPANCY_REF}) -> {spancy_shift_spatial.__file__}')

In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

import spancy_shift_spatial as ss

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 0
ALPHA = 0.6
print(f'Device: {DEVICE}')

## 2. Data

In [ ]:
DATA_URL = 'https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad'
DATA_PATH = 'PRAD_anndata.h5ad'


def download(url, path):
    # Downloads to path + '.part' first, so an interrupted download is never taken as complete.
    try:
        from tqdm.auto import tqdm
        bar = tqdm(unit='B', unit_scale=True, unit_divisor=1024, desc=path)

        def hook(blocks, block_size, total):
            if total > 0 and bar.total != total:
                bar.total = total
            bar.update(blocks * block_size - bar.n)
    except ImportError:
        bar = None

        def hook(blocks, block_size, total):
            if total > 0 and blocks % 2000 == 0:
                print(f'  {min(blocks * block_size, total) / 1e6:.0f} / {total / 1e6:.0f} MB')
    urllib.request.urlretrieve(url, path + '.part', reporthook=hook)
    if bar is not None:
        bar.close()
    os.replace(path + '.part', path)


if not os.path.exists(DATA_PATH):
    print('Downloading PRAD-CyCIF from Zenodo ...')
    download(DATA_URL, DATA_PATH)
print(f'{DATA_PATH}: {os.path.getsize(DATA_PATH) / 1e6:.0f} MB')
adata = ss.load_adata(DATA_PATH)
print(adata)

## 3. Stage 1 and region graph

In [ ]:
# Stage 1 (per-marker shift toward a KL-medoid reference sample) + scaling + region k-NN graph.
t0 = time.time()
prep = ss.prepare(adata)
print(f'\nprepare: {time.time() - t0:.0f} s')
print('bimodal markers (kept at their Stage 1 values):',
      [m for m, b in zip(adata.var_names, prep.is_bimodal) if b])
print('regions:', len(np.unique(prep.regions)))

## 4. Stage 2 training

In [ ]:
# Stage 2 training with the module defaults (10 epochs; Huber 0.1, NT-Xent 0.5, adversarial 0.3, MMD 1.0).
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
t0 = time.time()
model, history = ss.train(prep, device_str=DEVICE)
print(f'\ntraining: {(time.time() - t0) / 60:.1f} min')

## 5. Inference

In [ ]:
# Inference: the correction is computed once; any alpha can then be applied without re-running.
t0 = time.time()
delta = ss.correct(prep, model, device_str=DEVICE)
X_out = ss.apply(prep, delta, alpha=ALPHA)
print(f'correction: {time.time() - t0:.0f} s')

adata_out = prep.adata_base.copy()          # raw in X, Stage 1 in layers['normalized_base']
adata_out.layers['normalized'] = X_out      # Stage 1 + alpha * correction
X_s1_log = np.log1p(np.asarray(adata_out.layers['normalized_base']))
X_out_log = np.log1p(X_out)
uni = ~prep.is_bimodal
print(f'mean |change| vs Stage 1 (log1p): unimodal {np.abs(X_out_log[:, uni] - X_s1_log[:, uni]).mean():.4f}, '
      f'bimodal {np.abs(X_out_log[:, ~uni] - X_s1_log[:, ~uni]).mean():.4f} (0 by construction)')
print(adata_out)

## 6. Histograms — Raw | Stage 1 | SpaNCy-Shift

One curve per sample (tab20), log1p space; bin edges per marker from the raw data. Batch effects show
as curves that are shifted against each other in the raw column; after normalization the samples'
curves should line up while each keeps its shape. Bimodal markers are identical in the last two columns.

In [ ]:
import gc

RAW = '__RAW__'
N_HIST_BINS = 100
MARKERS_PER_FIG = 4
HIST_LAYERS = [(RAW, 'Raw'), ('normalized_base', 'Stage 1'), ('normalized', f'SpaNCy-Shift (α={ALPHA})')]
adata_norm = adata_out
marker_names = list(adata_out.var_names)
X_raw = adata_out.X.toarray() if hasattr(adata_out.X, 'toarray') else np.asarray(adata_out.X)
X_raw_log = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)

M_h = len(marker_names)
sample_ids_h = adata_norm.obs['sample_id'].values
samples_h = sorted(np.unique(sample_ids_h).tolist())
sidx_h = {s: np.where(sample_ids_h == s)[0] for s in samples_h}
colors_h = plt.cm.tab20(np.linspace(0, 1, len(samples_h)))
edges_h = {}
for mi in range(M_h):
    v = X_raw_log[:, mi]
    hi = float(np.percentile(v, 99.9))
    edges_h[mi] = np.linspace(float(v.min()), hi if hi > v.min() else float(v.max()), N_HIST_BINS + 1)

dens_h = {}
for key, _ in HIST_LAYERS:
    XL = X_raw_log if key == RAW else np.log1p(np.clip(np.asarray(adata_norm.layers[key]), 0, None)).astype(np.float32)
    for mi in range(M_h):
        for s in samples_h:
            dens_h[(key, mi, s)], _ = np.histogram(XL[sidx_h[s], mi], bins=edges_h[mi], density=True)
    del XL
    gc.collect()
ymax_h = {mi: max(d.max() for (k, mm, s), d in dens_h.items() if mm == mi) for mi in range(M_h)}


def _plot_rows(marker_idx):
    fig, axes = plt.subplots(len(marker_idx), len(HIST_LAYERS),
                             figsize=(2.9 * len(HIST_LAYERS), 2.3 * len(marker_idx)), squeeze=False, sharex='row')
    for r, mi in enumerate(marker_idx):
        ctr = 0.5 * (edges_h[mi][:-1] + edges_h[mi][1:])
        for c, (key, label) in enumerate(HIST_LAYERS):
            ax = axes[r, c]
            for s, col in zip(samples_h, colors_h):
                ax.plot(ctr, dens_h[(key, mi, s)], color=col, lw=0.8)
            ax.set_ylim(0, ymax_h[mi] * 1.05)
            if r == 0:
                ax.set_title(label, fontsize=9)
            if c == 0:
                ax.set_ylabel(marker_names[mi], fontsize=9)
            ax.tick_params(labelsize=7)
    fig.tight_layout()
    return fig


for p in range(0, M_h, MARKERS_PER_FIG):
    _plot_rows(list(range(p, min(p + MARKERS_PER_FIG, M_h))))
    plt.show()

## 7. Save the normalized `.h5ad`

In [ ]:
OUT_PATH = 'PRAD_normalized.h5ad'
adata_out.write_h5ad(OUT_PATH)
print(f'Saved {OUT_PATH} ({os.path.getsize(OUT_PATH) / 1e6:.0f} MB): X = raw, '
      f"layers['normalized_base'] = Stage 1, layers['normalized'] = SpaNCy-Shift (alpha={ALPHA})")
try:
    from google.colab import files
    files.download(OUT_PATH)
except ImportError:
    pass